# Machine Learning for Credit Risk Modeling: Predicting Customer Loan Default Using Financial and Behavioral Data

### Ploblem statement

Financial institutions face significant challenges in identifying loan applicants who are likely to default while ensuring that creditworthy customers are not unfairly rejected. Traditional credit risk assessment methods may fail to capture complex relationships within large and diverse financial datasets, leading to poor lending decisions and increased financial losses. This project aims to develop a machine learning model that predicts the probability of customer loan default using the Home Credit Default Risk dataset. By leveraging customer demographic information, financial history, credit bureau records, previous loan applications, installment payments, credit card activity, and POS/Cash loan history, the model seeks to improve credit risk assessment and support more accurate, data-driven lending decisions.

## Importing Libraries

In [1]:
#  Core data handling 
import os
import gc
import itertools
import random
import numpy as np
import pandas as pd

#  Data source 
from datasets import load_dataset

# Visualization 
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Model selection / splitting 
from sklearn.model_selection import train_test_split, RandomizedSearchCV

#  Feature selection 
from sklearn.feature_selection import mutual_info_classif, RFE

#  Models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

#  Evaluation metrics 
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, confusion_matrix, classification_report,accuracy_score, roc_curve
)

# Interpretability 
import shap

#  Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

c:\Users\User\Desktop\final project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Downloading the raw data

Pulls all 8 tables from the Hugging Face mirror of the competition. Saves each as a CSV into the `data/` folder.

In [2]:

# Make sure the data folder exists

folder = "data"
os.makedirs(folder, exist_ok=True)

# Home Credit dataset configurations

configs = {
    "application_train_dated": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "POS_CASH_balance": "POS_CASH_balance.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "installments_payments": "installments_payments.csv",
    "sample_submission": "sample_submission.csv"
}

print("=" * 60)
print("Downloading Home Credit Default Risk Dataset")
print("=" * 60)

for config_name, output_file in configs.items():

    print(f"\nLoading: {config_name}")

    try:
        dataset = load_dataset(
            "mohameddhameem/home-credit-default-risk",
            config_name
        )

         # Convert the HF dataset split to a pandas DataFrame
        df = dataset["train"].to_pandas()

         # Save to CSV so the rest of the notebook can work with plain pandas
        save_path = os.path.join("data", output_file)
        df.to_csv(save_path, index=False)

        print(f"✓ Saved: {output_file}")
        print(f"  Shape: {df.shape}")

        # Free memory before moving to the next (large) table
        del dataset
        del df
        gc.collect()

    except Exception as e:
        print(f"✗ Failed: {config_name}")
        print(e)

print("\n" + "=" * 60)
print("Download Complete!")
print("=" * 60)


Loading: application_train_dated


✓ Saved: application_train.csv
  Shape: (307511, 123)

Loading: application_test
✓ Saved: application_test.csv
  Shape: (48744, 121)

Loading: bureau
✓ Saved: bureau.csv
  Shape: (1716428, 17)

Loading: bureau_balance
✓ Saved: bureau_balance.csv
  Shape: (27299925, 3)

Loading: previous_application
✓ Saved: previous_application.csv
  Shape: (1670214, 37)

Loading: POS_CASH_balance
✓ Saved: POS_CASH_balance.csv
  Shape: (10001358, 8)

Loading: credit_card_balance
✓ Saved: credit_card_balance.csv
  Shape: (3840312, 23)

Loading: installments_payments
✓ Saved: installments_payments.csv
  Shape: (13605401, 8)

Loading: sample_submission
✓ Saved: sample_submission.csv
  Shape: (48744, 2)

Download Complete!


Due to account verification issues with Kaggle, I was unable to access the dataset directly through the Kaggle API. As an alternative, I downloaded the Home Credit Default Risk dataset from the Hugging Face repository, which mirrors the original Kaggle competition dataset. This ensured I could proceed with the project while using the same data structure and features as the original competition dataset.

In [3]:
# Load the primary applicant-level table and take a first look
train = pd.read_csv("data/application_train.csv")

print("Shape:", train.shape)
train.head()

Shape: (307511, 123)


,SK_ID_CURR,application_date,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,2018-01-01,1,Cash loans,M,N,Y,0,202500.0,406597.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,2018-01-01,0,Cash loans,F,N,N,0,270000.0,1293502.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,2018-01-01,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,2018-01-01,0,Cash loans,F,N,Y,0,135000.0,312682.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,2018-01-01,0,Cash loans,M,N,Y,0,121500.0,513000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


Loaded my training dataset into Python to give me a quick overview of what it contains.

Do a data validation and integrity check. Before I begin the analysis, I need to  verify that all the required dataset files are present, appear complete, and contain the expected columns.

In [4]:
folder = "data"

expected_files = [
    "application_train.csv", "application_test.csv", "bureau.csv",
    "bureau_balance.csv", "previous_application.csv", "POS_CASH_balance.csv",
    "credit_card_balance.csv", "installments_payments.csv"
]

missing = [f for f in expected_files if not os.path.exists(os.path.join(folder, f))]
if missing:
    print(f"✗ MISSING: {missing}")
else:
    print(f"All {len(expected_files)} expected files found in '{folder}'")

check = pd.read_csv(os.path.join(folder, "application_train.csv"), nrows=5)
print("TARGET column present:", "TARGET" in check.columns)

All 8 expected files found in 'data'
TARGET column present: True


In [5]:

files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "POS_CASH_balance.csv",
    "credit_card_balance.csv",
    "installments_payments.csv"
]

for file in files:
    df = pd.read_csv(f"data/{file}")
    print(f"{file}: {len(df):,} rows")

application_train.csv: 307,511 rows
application_test.csv: 48,744 rows
bureau.csv: 1,716,428 rows
bureau_balance.csv: 27,299,925 rows
previous_application.csv: 1,670,214 rows
POS_CASH_balance.csv: 10,001,358 rows
credit_card_balance.csv: 3,840,312 rows
installments_payments.csv: 13,605,401 rows


Before I start working with the data, I run two quick checks.

First, I make sure all 8 files are actually there, and that the main file has the TARGET column - that's the column that tells me who defaulted and who didn't, so if it's missing, nothing else in the project can work.

Second, I open each file and count how many rows it has. This helps me see how big each table is before I start combining them. For example, bureau_balance has over 27 million rows - way more than one row per person - which is why I had to shrink it down to one row per applicant before I could use it.

Basically, these two checks make sure everything is in place before I start the real work, so if something's wrong, I catch it early instead of getting a confusing error later on."

TARGET is a column with one value per applicant, and it only has two possible values:

0 = did not default → they repaid the loan

1 = did default → they failed to repay

##  Feature Engineering: Aggregate the Secondary Tables

There are five extra tables, and each one contains many records for the same person, such as information about past loans, credit history, or payment behaviour. Since the main application table contains only one row per person, these secondary tables need to be summarized into one row per applicant before they can be merged.

To do this, we use normal aggregation methods such as the average, total, minimum, maximum, and count to summarize each applicant's history. We also use time-aware aggregations to give more importance to recent behaviour. This includes recency-weighted averages, where recent records have a greater influence than older records, and recent time windows, which focus only on an applicant's behaviour during periods such as the last 6, 12, or 24 months.

These summaries are then merged with the main application table using the applicant's ID. After each merge, we check that the number of rows in the main table has not changed. This helps ensure that each applicant still has only one row and that no duplicate rows are created during the merging process.

In [6]:

# Define the folder where the input datasets are stored
INPUT_DIR = './data'

# Define the folder where processed files or outputs will be saved
OUTPUT_DIR = './output'

# Create the output folder if it does not already exist
# exist_ok=True prevents an error if the folder already exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def downcast(df, exclude=('SK_ID_CURR',)):
    """Shrink dtypes to reduce memory footprint (float64->float32, int64->smallest int, etc)."""

    # Loop through every column in the da   taframe one at a time
    for col in df.columns:

        if col in exclude:       # Skip columns listed in the exclude parameter
            continue

        if df[col].dtype == 'float64':             # Convert float64 columns to float32 This cuts memory usage roughly in half while maintaining good precision
            df[col] = df[col].astype('float32')

        elif df[col].dtype == 'int64':     # Convert int64 columns to the smallest possible integer type (e.g., int8, int16, int32) depending on the values stored
            df[col] = pd.to_numeric(df[col], downcast='integer')

        # If it's True/False, store it as a tiny integer (0 or 1) instead
        elif df[col].dtype == 'bool':
            df[col] = df[col].astype('int8')

        # If it's text, store it as a "category" type instead of plain text    
        elif df[col].dtype == object:
            df[col] = df[col].astype('category')
            
    return df

The function above helps in reducing memory usage, which is particularly important when working with large datasets, while keeping the data suitable for analysis and machine learning.

In [7]:
def recency_weight(days_since, halflife):
    """
    Exponential recency weight, vectorised.
    days_since : how many days/months ago the record happened (>= 0, 0 = today).
    halflife   : after this many days/months, weight drops to 0.5; after 2x, to 0.25, etc.
    """
    return np.power(0.5, days_since / halflife)


def weighted_group_mean(df, group_col, value_col, weight_col):
    """
    Vectorised, NaN-safe weighted mean of value_col per group_col, using weight_col.
    Equivalent to df.groupby(group_col).apply(lambda g: np.average(g[value_col],
    weights=g[weight_col])) but implemented as a groupby().sum() so it scales to
    multi-million-row tables instead of looping row by row.
    """
    valid = df[value_col].notna()
    w = df[weight_col].where(valid, 0.0)
    wx = df[value_col].fillna(0) * w
    tmp = pd.DataFrame({group_col: df[group_col], "_w": w, "_wx": wx})
    g = tmp.groupby(group_col).sum()
    out = g["_wx"] / g["_w"].replace(0, np.nan)
    out.name = None
    return out


The Home Credit dataset contains millions of records across several tables, and each applicant may have many historical records. Because older records may not be as relevant as recent ones, these two functions are created to make the feature engineering time-aware. Instead of treating every historical record as equally important, they give more weight to recent behaviour.

The recency_weight function calculates how important a record should be based on how long ago it happened. A recent record receives a higher weight, while an older record receives a lower weight. The halflife controls how quickly the weight decreases. For example, with a halflife of 12 months, a record from today has a weight of 1, a record from 12 months ago has a weight of 0.5, and a record from 24 months ago has a weight of 0.25.

The weighted_group_mean function then uses these weights to calculate a weighted average for each applicant. This means that recent values have more influence on the applicant's final average than older values. It also handles missing values safely and uses groupby().sum() rather than processing each applicant one at a time, making it much faster and more suitable for the millions of records in the Home Credit dataset.

Together, these functions help create features that capture recent customer behaviour, rather than relying only on lifetime averages that may give old and recent behaviour the same importance.

#### A. Aggregate **bureau_balance** - one row per **SK_ID_BUREAU**

This table is monthly loan status history at *other* credit institutions, keyed by ***SK_ID_BUREAU*** (not ***SK_ID_CURR***), so it has to be aggregated up one level before it can attach to ***bureau***.

In [8]:
# Read the CSV file using optimized data types to reduce memory usage.
# bureau_balance is a largest table in the project (~27 million rows), so specifying smaller data types saves a significant amount of RAM.
bb = pd.read_csv(
    f"{INPUT_DIR}/bureau_balance.csv",
    dtype={"SK_ID_BUREAU": "int32", "MONTHS_BALANCE": "int16", "STATUS": "category"},   # Tell pandas exactly what size/type to use for each column while loading,
    # instead of letting it guess, this saves a lot of memory right from the start
)
print("Loaded bureau_balance:", bb.shape)

# Convert each repayment status into separate binary columns.
# One-hot encode the monthly status flag (0-5 = days-past-due bucket, C = closed, X = unknown)
# This turns the single STATUS column into several 0/1 columns, one per status value
status_dummies = pd.get_dummies(bb["STATUS"], prefix="BB_STATUS")
# Keep only the required columns and combine them with the new dummy variables
bb = pd.concat([bb[["SK_ID_BUREAU", "MONTHS_BALANCE"]], status_dummies], axis=1)

# Aggregate to one row per SK_ID_BUREAU: how many months of history, its span, and how many
# months fell into each status bucket

# Aggregate Monthly Records
# Each loan (SK_ID_BUREAU) has many monthly records. summarize them into a single row
bb_agg = bb.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS_COUNT=("MONTHS_BALANCE", "count"),        # Total number of months recorded
    BB_MONTHS_MIN=("MONTHS_BALANCE", "min"),            # Earliest month available
    BB_MONTHS_MAX=("MONTHS_BALANCE", "max"),            # Most recent month available
)

# Sum each dummy column.
# Since the dummy columns contain only 0 and 1, summing gives the number of months in each status.
bb_agg = bb_agg.join(bb.groupby("SK_ID_BUREAU")[status_dummies.columns].sum())

# Convert the grouped index (loan ID) back into a normal column
bb_agg.reset_index(inplace=True)

# Display the aggregated dataset
print("Aggregated bureau_balance shape:", bb_agg.shape)
# Store the summarized dataset as a Parquet file. Parquet files are smaller, faster to read, and more efficient than CSV files.
bb_agg.to_parquet(f"{OUTPUT_DIR}/agg_bureau_balance.parquet", index=False)

# Delete large objects that are no longer needed. ( The original dataset is very large and is no longer needed after aggregation. Deleting it frees up memory for the next processing steps.)
del bb, status_dummies
# Force Python's garbage collector to release memory.
gc.collect()

Loaded bureau_balance: (27299925, 3)
Aggregated bureau_balance shape: (817395, 12)


0

This table (bureau_balance) is huge - about 27 million rows - and it tracks the monthly status of loans or credits that applicants had with other financial institutions. Each row represents one month of one loan's history.

Since the table is so large, the code loads it carefully by telling pandas the exact, smaller data type for each column upfront, instead of loading everything using larger default data types and reducing them afterward. This helps save memory from the moment the data is loaded.

Each loan's monthly status is represented by a code such as "0" (current/no payment problems), "1-5" (increasing levels of overdue payments), "C" (closed), or "X" (unknown). The code converts this single status column into several 0/1 columns, one for each possible status, so that the number of months spent in each status can be counted.

It then groups the records by loan ID and reduces each loan to one summary row. The summary shows how many months of history are available, the earliest and latest month recorded, and how many months the loan spent in each status.

Finally, the smaller summary table is saved to disk as a Parquet file, and the large original table is removed from memory because it is no longer needed. This helps prevent the notebook from running out of memory when processing the other large tables.

#### B. Aggregate **bureau** (+ bureau_balance features) - one row per **SK_ID_CURR**

Past loans at other credit institutions. Each applicant can have many rows here (many past loans), so we one-hot the categoricals, fold in the ***bureau_balance*** aggregates, then group by ***SK_ID_CURR*** with mean/sum/min/max.

In [9]:
# Load the bureau dataset

# Read the bureau.csv file into a DataFrame.
# This dataset contains information about each applicant's previous loans from external credit bureaus.bureau = pd.read_csv(f"{INPUT_DIR}/bureau.csv")
bureau = pd.read_csv(f"{INPUT_DIR}/bureau.csv")

print("Loaded bureau:", bureau.shape)

# Merge with the aggregated bureau_balance dataset. Combine bureau.csv with the aggregated bureau_balance
# using SK_ID_BUREAU as the common key.
# how="left" ensures that every bureau loan is kept, even if it has no matching bureau_balance record.

bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

# One-hot encode the categorical columns (Convert each category into separate binary columns.)
# This allows machine learning algorithms to use categorical variables.
for col in ["CREDIT_ACTIVE", "CREDIT_CURRENCY", "CREDIT_TYPE"]:
    bureau = pd.concat([bureau, pd.get_dummies(bureau[col], prefix=col)], axis=1)

# Select Numeric Columns for Aggregation
# Numeric columns get mean/sum/min/max per applicant (captures typical + extreme loan behavior)
num_cols = [
    # Days since the credit was granted
    "DAYS_CREDIT",
    # Number of overdue days
    "CREDIT_DAY_OVERDUE",
    # Expected credit end date
    "DAYS_CREDIT_ENDDATE",
    # Actual credit closing date
    "DAYS_ENDDATE_FACT",
    # Maximum overdue credit amount
    "AMT_CREDIT_MAX_OVERDUE",
    # Number of times the loan was prolonged
    "CNT_CREDIT_PROLONG",
    # Total credit amount
    "AMT_CREDIT_SUM",
    # Outstanding debt
    "AMT_CREDIT_SUM_DEBT",
    # Credit limit
    "AMT_CREDIT_SUM_LIMIT",
    # Amount currently overdue
    "AMT_CREDIT_SUM_OVERDUE",
    # Last bureau update
    "DAYS_CREDIT_UPDATE",
    # Loan annuity
    "AMT_ANNUITY",
    # Aggregated bureau_balance variables
    "BB_MONTHS_COUNT",
    "BB_MONTHS_MIN",
    "BB_MONTHS_MAX",
]
# Find the names of the new  columns we just created above, to summarize them too
bb_status_cols = [c for c in bureau.columns if c.startswith("BB_STATUS_")]
dummy_cols = [c for c in bureau.columns if c.startswith(("CREDIT_ACTIVE_", "CREDIT_CURRENCY_", "CREDIT_TYPE_"))]

# Define Aggregation Rules
# For every numeric column calculate:Mean, Sum, Minimum, Maximum
agg_dict = {c: ["mean", "sum", "min", "max"] for c in num_cols + bb_status_cols}
# For dummy variables calculate only the mean. Example:CREDIT_ACTIVE_Active Mean = proportion of loans that are Active.
agg_dict.update({c: "mean" for c in dummy_cols})     
# Count the total number of bureau loans each applicant has. 
agg_dict["SK_ID_BUREAU"] = "count" 

# Aggregate to One Row Per Customer
# Group all rows by person (applicant ID), and apply the summary plan above
bureau_agg = bureau.groupby("SK_ID_CURR").agg(agg_dict)
# Rename the resulting columns to be clear and consistent
bureau_agg.columns = ["BUREAU_" + "_".join(col).upper() for col in bureau_agg.columns]
# Turn the applicant ID from an index back into a normal column
bureau_agg.reset_index(inplace=True)

# ---- Time-aware / recency features (bureau) ----
# DAYS_CREDIT is negative: 0 = credit opened today, more negative = older.
# Flip the sign so "age" reads as a normal, non-negative day count.
bureau["CREDIT_AGE_DAYS"] = -bureau["DAYS_CREDIT"]

# A loan opened today gets weight 1.0; one opened 365 days ago gets weight 0.5;
# one opened 730 days ago gets weight 0.25, and so on.
BUREAU_HALFLIFE_DAYS = 365
bureau["RECENCY_W"] = recency_weight(bureau["CREDIT_AGE_DAYS"].clip(lower=0), BUREAU_HALFLIFE_DAYS)

# Recency-weighted average debt/overdue - recent loans influence this more than old ones.
recency_cols = ["AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CREDIT_DAY_OVERDUE"]
bureau_recency = pd.DataFrame({
    f"BUREAU_RECENCY_WEIGHTED_{col}": weighted_group_mean(bureau, "SK_ID_CURR", col, "RECENCY_W")
    for col in recency_cols
})

# "Last 2 years only" flat aggregates - a default from 8 years ago no longer drags
# down an applicant whose bureau history has been clean for the last 2 years.
recent_bureau = bureau[bureau["CREDIT_AGE_DAYS"] <= 730]
bureau_l2y = recent_bureau.groupby("SK_ID_CURR").agg(
    BUREAU_L2Y_COUNT=("SK_ID_BUREAU", "count"),
    BUREAU_L2Y_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
    BUREAU_L2Y_DEBT_MAX=("AMT_CREDIT_SUM_DEBT", "max"),
    BUREAU_L2Y_OVERDUE_MEAN=("AMT_CREDIT_SUM_OVERDUE", "mean"),
    BUREAU_L2Y_OVERDUE_MAX=("AMT_CREDIT_SUM_OVERDUE", "max"),
    BUREAU_L2Y_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
)

# Most recent bureau loan only - the applicant's *current* standing, not a lifetime
# average. credit_active_dummies reuses the CREDIT_ACTIVE_* one-hot columns already
# created above, so the most recent loan's Active/Closed/Bad debt status comes along too.
credit_active_dummies = [c for c in dummy_cols if c.startswith("CREDIT_ACTIVE_")]
last_loan_cols = (
    ["CREDIT_AGE_DAYS", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE", "CREDIT_DAY_OVERDUE"]
    + credit_active_dummies
)
most_recent_bureau = (
    bureau.sort_values("CREDIT_AGE_DAYS")
    .groupby("SK_ID_CURR")
    .first()[last_loan_cols]
    .add_prefix("BUREAU_LAST_LOAN_")
)

# Fold all the new time-aware features into the existing flat aggregate before saving.
bureau_agg = (
    bureau_agg.set_index("SK_ID_CURR")
    .join(bureau_recency)
    .join(bureau_l2y)
    .join(most_recent_bureau)
    .reset_index()
)

print("Aggregated bureau shape:", bureau_agg.shape)
bureau_agg.to_parquet(f"{OUTPUT_DIR}/agg_bureau.parquet", index=False)

del bureau, bb_agg
gc.collect()


Loaded bureau: (1716428, 17)
Aggregated bureau shape: (305811, 134)


0

The bureau table contains information about each applicant's previous loans or credits from external financial institutions. One applicant can appear many times because they may have had several previous loans, so the goal is to summarize all of these records into one row per applicant.

First, the code loads the bureau table and attaches the monthly history summary from bureau_balance by matching the loans using SK_ID_BUREAU. This means each previous loan now also has information such as how many months of history are available and how often it appeared in different payment-status categories.

The code then converts categorical information, such as whether a credit is Active, Closed, or Sold, into separate 0/1 columns. This allows these categories to be included when calculating summary statistics. The numerical columns are then grouped by applicant and summarized using the mean, sum, minimum, and maximum. This captures both the applicant's typical credit behaviour and extreme situations, such as their largest outstanding debt or highest overdue amount. For the categorical 0/1 columns, the mean represents the proportion of the applicant's loans belonging to each category. The code also counts the total number of previous loans each applicant has.

The code then adds time-aware features so that recent credit behaviour is not treated exactly the same as very old behaviour. It calculates how old each loan is and gives more weight to recent loans when calculating weighted averages of debt, overdue amounts, and overdue days. It also creates last-two-year features, which focus only on loans from the most recent two years. In addition, it identifies the applicant's most recent bureau loan to capture their latest credit standing rather than relying only on their lifetime history.

Finally, all of these features - the standard lifetime aggregates, recency-weighted features, recent two-year features, and most-recent-loan information - are combined into one applicant-level table. The final table is saved as a Parquet file, and the large datasets that are no longer needed are removed from memory to reduce RAM usage.

#### C. Aggregate **POS_CASH_balance** - one row per **SK_ID_CURR**

Monthly balance history on POS/cash loans. This table already carries ***SK_ID_CURR*** directly, so no intermediate ***SK_ID_PREV*** step is needed.

In [10]:
# Load the POS_CASH_balance dataset

# Read the dataset while specifying smaller data types to reduce memory usage.
pos = pd.read_csv(
    f"{INPUT_DIR}/POS_CASH_balance.csv",
    dtype={
        "SK_ID_PREV": "int32",                 # Previous loan ID
        "SK_ID_CURR": "int32",                 # Customer ID
        "MONTHS_BALANCE": "int16",             # Months before current application
        "CNT_INSTALMENT": "float32",           # Total scheduled installments
        "CNT_INSTALMENT_FUTURE": "float32",    # Remaining installments
        "NAME_CONTRACT_STATUS": "category",    # Loan status
        "SK_DPD": "int32",                     # Days Past Due
        "SK_DPD_DEF": "int32",                 # Days Past Due with tolerance
    },
)
print("Loaded POS_CASH_balance:", pos.shape)

# One-hot encode contract status (Active, Completed, Signed, etc.)
status_dummies = pd.get_dummies(pos["NAME_CONTRACT_STATUS"], prefix="POS_STATUS")
# Remove the original status column and append the newly created dummy columns.
pos = pd.concat([pos.drop(columns=["NAME_CONTRACT_STATUS"]), status_dummies], axis=1)

# Define Aggregation Rules

# For numerical variables calculate:Mean,Maximum
agg_dict = {c: ["mean", "max"] for c in ["CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE", "SK_DPD", "SK_DPD_DEF"]}

# For dummy variables calculate the mean. The mean represents the proportion of loans in each contract status.
agg_dict.update({c: "mean" for c in status_dummies.columns})

# Count the number of unique previous POS/Cash loans belonging to each customer.
agg_dict["SK_ID_PREV"] = "nunique" 

# Aggregate to One Row Per Customer
# Group all POS/Cash loans belonging to each customer.
pos_agg = pos.groupby("SK_ID_CURR").agg(agg_dict)
# Rename Columns
pos_agg.columns = ["POS_" + "_".join(col).upper() for col in pos_agg.columns]

# Convert SK_ID_CURR back into a normal column.
pos_agg.reset_index(inplace=True)

# ---- Time-aware / recency features (POS_CASH_balance) ----
# MONTHS_BALANCE is negative: 0 = the month closest to the application, more
# negative = further in the past. Flip sign for an intuitive "months ago".
pos["MONTHS_AGO"] = -pos["MONTHS_BALANCE"]

POS_HALFLIFE_MONTHS = 6
pos["RECENCY_W"] = recency_weight(pos["MONTHS_AGO"].clip(lower=0), POS_HALFLIFE_MONTHS)

pos_recency_feats = pd.DataFrame({
    "POS_RECENCY_WEIGHTED_DPD": weighted_group_mean(pos, "SK_ID_CURR", "SK_DPD", "RECENCY_W"),
    "POS_RECENCY_WEIGHTED_DPD_DEF": weighted_group_mean(pos, "SK_ID_CURR", "SK_DPD_DEF", "RECENCY_W"),
})

# Last-6-months and last-12-months only - does recent POS/cash behaviour look
# worse than the applicant's full history?
pos_l6m = pos[pos["MONTHS_AGO"] <= 6].groupby("SK_ID_CURR").agg(
    POS_L6M_DPD_MEAN=("SK_DPD", "mean"), POS_L6M_DPD_MAX=("SK_DPD", "max")
)
pos_l12m = pos[pos["MONTHS_AGO"] <= 12].groupby("SK_ID_CURR").agg(
    POS_L12M_DPD_MEAN=("SK_DPD", "mean"), POS_L12M_DPD_MAX=("SK_DPD", "max")
)

pos_agg = (
    pos_agg.set_index("SK_ID_CURR")
    .join(pos_recency_feats)
    .join(pos_l6m)
    .join(pos_l12m)
    .reset_index()
)

# Trend: is the last 6 months of DPD worse than the applicant's overall average?
# Positive = payment behaviour has been getting worse lately, not just historically bad.
pos_agg["POS_DPD_TREND"] = pos_agg["POS_L6M_DPD_MEAN"] - pos_agg["POS_SK_DPD_MEAN"]

print("Aggregated POS_CASH_balance shape:", pos_agg.shape)
pos_agg.to_parquet(f"{OUTPUT_DIR}/agg_pos_cash.parquet", index=False)

del pos, status_dummies
gc.collect()

Loaded POS_CASH_balance: (10001358, 8)
Aggregated POS_CASH_balance shape: (337252, 26)


0

First, the code loads the large dataset using smaller data types to reduce memory usage. It then converts the loan status, such as Active or Completed, into separate 0/1 columns so these categories can be included in the calculations. The code then groups all records belonging to the same customer and calculates useful summary features, including the average and maximum number of installments, average and maximum days past due, the proportion of loans in each contract status, and the number of unique POS/Cash loans.

The code also adds time-aware features because recent repayment behaviour can be more important than older behaviour. It gives more weight to recent records when calculating days past due, and it calculates the customer's repayment behaviour over the last 6 and 12 months. This allows us to see what the customer's recent payment behaviour looks like instead of relying only on their entire history.

Finally, the code creates a payment behaviour trend by comparing the customer's average days past due during the last 6 months with their overall historical average. A positive value means that the customer has been more overdue recently than they were on average in the past, which may indicate that their repayment behaviour is getting worse.

The final summarized table is then saved as a Parquet file, and the large original dataset is deleted from memory to keep the processing efficient and reduce the risk of running out of RAM.

#### D. Aggregate **credit_card_balance** -  one row per **SK_ID_CURR**

In [11]:

# Load the Credit Card Balance Dataset


# Read the credit_card_balance.csv file.
# This dataset contains monthly information about customers'credit card accounts.

cc = pd.read_csv(f"{INPUT_DIR}/credit_card_balance.csv")

# Display the number of rows and columns loaded
print("Loaded credit_card_balance:", cc.shape)


# One-Hot Encode the Contract Status

# Convert each contract status (e.g., Active, Completed) into binary (0/1) columns.

status_dummies = pd.get_dummies(
    cc["NAME_CONTRACT_STATUS"],
    prefix="CC_STATUS"
)

# Remove the original contract status column and append the dummy variables.

cc = pd.concat(
    [
        cc.drop(columns=["NAME_CONTRACT_STATUS"]),
        status_dummies
    ],
    axis=1
)

# Select Numeric Columns
# These are the numerical variables that describe customers' credit card usage and repayment behavior.

num_cols = [
    # Current outstanding balance
    "AMT_BALANCE",
    # Credit limit
    "AMT_CREDIT_LIMIT_ACTUAL",
    # ATM cash withdrawals
    "AMT_DRAWINGS_ATM_CURRENT",
    # Total withdrawals
    "AMT_DRAWINGS_CURRENT",
    # Other withdrawals
    "AMT_DRAWINGS_OTHER_CURRENT",
    # POS purchases
    "AMT_DRAWINGS_POS_CURRENT",
    # Minimum payment due
    "AMT_INST_MIN_REGULARITY",
    # Current payment
    "AMT_PAYMENT_CURRENT",
    # Total payment
    "AMT_PAYMENT_TOTAL_CURRENT",
    # Principal receivable
    "AMT_RECEIVABLE_PRINCIPAL",
    # Receivable amount
    "AMT_RECIVABLE",
    # Total receivable
    "AMT_TOTAL_RECEIVABLE",
    # Number of ATM withdrawals
    "CNT_DRAWINGS_ATM_CURRENT",
    # Total number of withdrawals
    "CNT_DRAWINGS_CURRENT",
    # Other withdrawals count
    "CNT_DRAWINGS_OTHER_CURRENT",
    # POS withdrawal count
    "CNT_DRAWINGS_POS_CURRENT",
    # Number of completed installments
    "CNT_INSTALMENT_MATURE_CUM",
    # Days past due
    "SK_DPD",
    # Days past due with tolerance
    "SK_DPD_DEF"
]

# Define Aggregation Rules

# For each numeric variable calculate:Mean ,Maximum, Sum

agg_dict = {
    c: ["mean", "max", "sum"]
    for c in num_cols
}

# For contract status dummy variables,calculate the mean (proportion).
agg_dict.update({c: "mean"for c in status_dummies.columns})

# Count the number of unique credit card accounts each customer has had.

agg_dict["SK_ID_PREV"] = "nunique"

# Aggregate to One Row Per Customer
# Group all credit card records belonging to each customer.

cc_agg = cc.groupby("SK_ID_CURR").agg(agg_dict)

# Rename Columns
# Create descriptive column names.
cc_agg.columns = ["CC_" + "_".join(col).upper()for col in cc_agg.columns]

# Convert SK_ID_CURR back into a normal column.
cc_agg.reset_index(inplace=True)

# Display Results
# ---- Time-aware / recency features (credit_card_balance) ----
cc["MONTHS_AGO"] = -cc["MONTHS_BALANCE"]

CC_HALFLIFE_MONTHS = 6
cc["RECENCY_W"] = recency_weight(cc["MONTHS_AGO"].clip(lower=0), CC_HALFLIFE_MONTHS)

# Recency-weighted credit utilization - recent months count more than months from
# years ago, which the flat mean elsewhere in this cell treats identically.
cc["UTILIZATION"] = cc["AMT_BALANCE"] / cc["AMT_CREDIT_LIMIT_ACTUAL"].replace(0, np.nan)
cc_recency_feats = pd.DataFrame({
    "CC_RECENCY_WEIGHTED_UTILIZATION": weighted_group_mean(cc, "SK_ID_CURR", "UTILIZATION", "RECENCY_W"),
    "CC_RECENCY_WEIGHTED_DPD": weighted_group_mean(cc, "SK_ID_CURR", "SK_DPD", "RECENCY_W"),
})

cc_l6m = cc[cc["MONTHS_AGO"] <= 6].groupby("SK_ID_CURR").agg(
    CC_L6M_BALANCE_MEAN=("AMT_BALANCE", "mean"), CC_L6M_DPD_MAX=("SK_DPD", "max")
)
cc_l12m = cc[cc["MONTHS_AGO"] <= 12].groupby("SK_ID_CURR").agg(
    CC_L12M_BALANCE_MEAN=("AMT_BALANCE", "mean"), CC_L12M_DPD_MAX=("SK_DPD", "max")
)

cc_agg = (
    cc_agg.set_index("SK_ID_CURR")
    .join(cc_recency_feats)
    .join(cc_l6m)
    .join(cc_l12m)
    .reset_index()
)

print("Aggregated credit_card_balance shape:", cc_agg.shape)

# Save the Aggregated Dataset
cc_agg.to_parquet(f"{OUTPUT_DIR}/agg_credit_card.parquet",index=False)

# Free Memory
# Delete objects that are no longer needed.
del cc, status_dummies
# Force Python to release unused memory.
gc.collect()

Loaded credit_card_balance: (3840312, 23)
Aggregated credit_card_balance shape: (103558, 72)


0

The credit_card_balance.csv table contains the month-by-month history of customers' credit card accounts, so the same customer can appear many times - once for each month of each credit card account. To prepare this information for machine learning, the code summarizes these records into one row per customer (SK_ID_CURR).

First, the code converts the credit card contract status, such as Active or Completed, into separate 0/1 dummy variables. It then groups all records belonging to the same customer and calculates the average, maximum, and total for important credit card features such as balances, credit limits, payments, withdrawals, receivables, and days past due. It also calculates the proportion of records in each contract status and counts the number of unique credit card accounts each customer has had.

The code then adds time-aware features so that recent credit card behaviour has more influence than older behaviour. It calculates credit utilization, which compares the customer's outstanding balance with their credit limit, and then creates a recency-weighted version where recent months have more influence. It also creates a recency-weighted measure of days past due.

In addition, the code looks specifically at the customer's last 6 and 12 months of credit card activity. This captures recent behaviour, such as the average balance and maximum days past due, rather than relying only on the customer's entire credit card history.

Finally, all these features are combined into the aggregated customer-level table, the column names are given clear CC_ prefixes, and the result is saved as a Parquet file. The original large dataset is then deleted from memory to free up RAM before processing the next table.

#### E. Aggregate **previous_application** - one row per **SK_ID_CURR**


In [12]:
# Load the Previous Application Dataset

# Read the previous_application.csv file. Each row represents one previous loan application made by a customer.
prev = pd.read_csv(f"{INPUT_DIR}/previous_application.csv")
print("Loaded previous_application:", prev.shape)

# In this dataset, the value 365243 is a placeholdermeaning "date not available".
# Replace it with NaN (missing value) so it is ignored during statistical calculations.
for col in ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]:
    prev[col] = prev[col].replace(365243, np.nan)

# One-hot encode all categorical columns, then take the per-applicant mean of each dummy
# (= "share of previous applications that had this category")
cat_cols = [
    "NAME_CONTRACT_TYPE", "FLAG_LAST_APPL_PER_CONTRACT", "NAME_CASH_LOAN_PURPOSE",
    "NAME_CONTRACT_STATUS", "NAME_PAYMENT_TYPE", "CODE_REJECT_REASON", "NAME_TYPE_SUITE",
    "NAME_CLIENT_TYPE", "NAME_GOODS_CATEGORY", "NAME_PORTFOLIO", "NAME_PRODUCT_TYPE",
    "CHANNEL_TYPE", "NAME_SELLER_INDUSTRY", "NAME_YIELD_GROUP", "PRODUCT_COMBINATION",
]
# Start with the customer ID.
dummy_frames = [prev[["SK_ID_CURR"]]]
# Store the names of all dummy columns.
dummy_cols = []
# Convert each categorical column into dummy variables.
for col in cat_cols:
    d = pd.get_dummies(prev[col], prefix=col)
    dummy_cols.extend(d.columns.tolist())
    dummy_frames.append(d)
# Combine all dummy variables into one DataFrame.
prev_dummies = pd.concat(dummy_frames, axis=1)

# Numeric columns get mean/max/min/sum
num_cols = [
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE",
    "RATE_DOWN_PAYMENT", "RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION",
    "SELLERPLACE_AREA", "CNT_PAYMENT", "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION", "NFLAG_INSURED_ON_APPROVAL",
]

# For every numeric column calculate:Mean, Maximum, Minimum, Sum
agg_dict = {c: ["mean", "max", "min", "sum"] for c in num_cols}
# Group all previous applications belonging to each customer.
prev_num_agg = prev.groupby("SK_ID_CURR").agg(agg_dict)
prev_num_agg.columns = ["PREV_" + "_".join(col).upper() for col in prev_num_agg.columns]

# Calculate the mean of every dummy column.
# This represents the proportion of previous applications in each category.
prev_dummy_agg = prev_dummies.groupby("SK_ID_CURR")[dummy_cols].mean()
# Rename columns.
prev_dummy_agg.columns = ["PREV_" + c.upper() + "_MEAN" for c in prev_dummy_agg.columns]
# Count how many previous applications each customer has submitted.
prev_count = prev.groupby("SK_ID_CURR")["SK_ID_PREV"].count().rename("PREV_COUNT")
# Merge All Aggregated Features
# Combine: Numeric summaries, Dummy summaries, Application count
prev_agg = prev_num_agg.join(prev_dummy_agg).join(prev_count)
# Convert SK_ID_CURR back into a normal column.
prev_agg.reset_index(inplace=True)
#  Display Results
# ---- Time-aware / recency features (previous_application) ----
# NAME_CONTRACT_STATUS_* dummies live in prev_dummies, not in prev itself (unlike the
# bureau/POS/credit_card cells) - pull the status columns onto prev so DAYS_DECISION-
# based recency logic can use them directly. Both frames share the same row index,
# so this is a safe, order-preserving join.
prev_status_cols_all = [c for c in dummy_cols if c.startswith("NAME_CONTRACT_STATUS_")]
prev = prev.join(prev_dummies[prev_status_cols_all])

# DAYS_DECISION is negative: 0 = decided today, more negative = longer ago.
prev["APP_AGE_DAYS"] = -prev["DAYS_DECISION"]

PREV_HALFLIFE_DAYS = 365
prev["RECENCY_W"] = recency_weight(prev["APP_AGE_DAYS"].clip(lower=0), PREV_HALFLIFE_DAYS)

# Recency-weighted refusal rate: a refusal 6 months ago should worry a lender far
# more than one from 6 years ago. The flat PREV_NAME_CONTRACT_STATUS_REFUSED_MEAN
# column above can't distinguish those two cases - this one can.
refused_col = "NAME_CONTRACT_STATUS_Refused"
prev_recency_feats = pd.DataFrame({
    "PREV_RECENCY_WEIGHTED_REFUSAL_RATE": weighted_group_mean(prev, "SK_ID_CURR", refused_col, "RECENCY_W"),
    "PREV_RECENCY_WEIGHTED_AMT_CREDIT": weighted_group_mean(prev, "SK_ID_CURR", "AMT_CREDIT", "RECENCY_W"),
})

# "Last 2 years only" flat aggregates.
recent_prev = prev[prev["APP_AGE_DAYS"] <= 730]
prev_l2y = recent_prev.groupby("SK_ID_CURR").agg(
    PREV_L2Y_COUNT=("SK_ID_PREV", "count"),
    PREV_L2Y_REFUSED_MEAN=(refused_col, "mean"),
    PREV_L2Y_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"),
)

# Most recent previous application only - its outcome (Approved/Refused/Canceled)
# and terms are a much sharper signal of an applicant's current standing with Home
# Credit than an average across every application they've ever submitted.
last_status_cols = [c for c in prev.columns if c.startswith("NAME_CONTRACT_STATUS_")]
last_app_cols = ["APP_AGE_DAYS", "AMT_CREDIT"] + last_status_cols
most_recent_prev = (
    prev.sort_values("APP_AGE_DAYS")
    .groupby("SK_ID_CURR")
    .first()[last_app_cols]
    .add_prefix("PREV_LAST_APP_")
)

prev_agg = (
    prev_agg.set_index("SK_ID_CURR")
    .join(prev_recency_feats)
    .join(prev_l2y)
    .join(most_recent_prev)
    .reset_index()
)

print("Aggregated previous_application shape:", prev_agg.shape)
# Save the Aggregated Dataset
prev_agg.to_parquet(f"{OUTPUT_DIR}/agg_previous_app.parquet", index=False)

# Free Memory

# Delete large DataFrames that are no longer needed.

del prev
del prev_dummies
del prev_num_agg
del prev_dummy_agg

# Release unused memory.

gc.collect()

Loaded previous_application: (1670214, 37)
Aggregated previous_application shape: (338857, 217)


0

The previous_application.csv table contains information about previous loan applications that customers submitted to Home Credit, unlike the bureau table, which contains credit information from other financial institutions. Since one customer can have many previous applications, the same customer can appear multiple times. The goal is therefore to summarize these applications into one row per customer (SK_ID_CURR).

First, the code handles a data-quality issue where some date columns use 365243 as a placeholder for a missing date. These values are replaced with NaN so that they do not distort calculations such as averages, minimums, and maximums.

Next, the categorical information, such as contract type, loan purpose, contract status, payment type, rejection reason, and customer type, is converted into 0/1 dummy variables. This allows the categorical information to be summarized numerically.

The code then creates the basic customer-level summary in three parts. For numerical features such as loan amounts, interest rates, installment information, and application timing, it calculates the mean, maximum, minimum, and total across all previous applications. For the dummy variables, it calculates the mean, which represents the proportion of the customer's previous applications belonging to each category. For example, it can show the proportion of applications that were refused or approved. It also counts the total number of previous applications submitted by each customer.

The code then adds time-aware features so that recent application behaviour receives more attention than very old applications. It uses DAYS_DECISION to determine how long ago each application was made and gives more weight to recent applications when calculating the refusal rate and credit amount. This helps distinguish between a customer who was refused recently and one whose refusal happened many years ago.

It also creates last-two-year features, which focus only on applications made within the previous two years. These capture recent application activity, refusal rates, and average credit amounts. In addition, the code identifies the most recent previous application, including its outcome, such as Approved, Refused, or Canceled, and its credit amount. This provides a more current view of the applicant's relationship with Home Credit than a lifetime average alone.

Finally, all the standard and time-aware features are combined into one customer-level table, the columns are given clear PREV_ prefixes, and the resulting dataset is saved as a Parquet file. The large intermediate DataFrames are then deleted from memory to keep the processing efficient.

#### F. Aggregate **installments_payments**

In [13]:

# Load the installments_payments table - the actual record of whether each loan installment was paid on time and in full
inst = pd.read_csv(f"{INPUT_DIR}/installments_payments.csv")
print("Loaded installments_payments:", inst.shape)

# Create New Payment Behaviour Features

# Calculate how many days late (or early) a payment was made.
# Positive value  -> payment was late.
# Zero            -> payment was made on time.
# Negative value  -> payment was made early.
inst["DAYS_LATE"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]       
# Calculate the difference between the expected installment amount and the amount actually paid.
# Positive value -> customer underpaid.
# Zero           -> exact payment.
# Negative value -> customer paid more than required.
inst["AMT_PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]      

# Calculate the payment ratio.
# 1.0  -> paid exactly the required amount.
# >1   -> overpaid.
# <1   -> underpaid.
# Replace zero installment values with NaN to avoid division-by-zero errors.   
inst["AMT_PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)

# List of number columns we want to summarize for each person
num_cols = [
    "NUM_INSTALMENT_VERSION", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT",
    "AMT_INSTALMENT", "AMT_PAYMENT", "DAYS_LATE", "AMT_PAYMENT_DIFF", "AMT_PAYMENT_RATIO",
]

# Define Aggregation Rules

# For every numeric feature calculate:, Mean, Maximum, Minimum, Sum, Standard deviation
agg_dict = {c: ["mean", "max", "min", "sum", "std"] for c in num_cols}
# Count the number of unique previous loans with installment records.
agg_dict["SK_ID_PREV"] = "nunique"       
# Count the total number of installment records.        
agg_dict["NUM_INSTALMENT_NUMBER"] = "count"      

# Aggregate to One Row Per Customer
# Group all installment records belongingto each customer.
inst_agg = inst.groupby("SK_ID_CURR").agg(agg_dict)
# Rename Columns

# Create descriptive column names beginning with INSTAL_.
inst_agg.columns = ["INSTAL_" + "_".join(col).upper() for col in inst_agg.columns]
# Convert SK_ID_CURR back into a normal column.
inst_agg.reset_index(inplace=True)

# Display Results
# ---- Time-aware / recency features (installments_payments) ----
# This table already carries DAYS_INSTALMENT / DAYS_LATE, which makes it the single
# best table in the project for recency-aware features: it is a payment-by-payment
# timeline, not just a lifetime summary.
inst["INST_AGE_DAYS"] = -inst["DAYS_INSTALMENT"]
inst["IS_LATE"] = (inst["DAYS_LATE"] > 0).astype("int8")

INSTAL_HALFLIFE_DAYS = 180
inst["RECENCY_W"] = recency_weight(inst["INST_AGE_DAYS"].clip(lower=0), INSTAL_HALFLIFE_DAYS)

# Recency-weighted lateness - a payment 3 months ago should move this number far
# more than one from 5 years ago, unlike the flat INSTAL_DAYS_LATE_MEAN above.
inst_recency_feats = pd.DataFrame({
    "INSTAL_RECENCY_WEIGHTED_DAYS_LATE": weighted_group_mean(inst, "SK_ID_CURR", "DAYS_LATE", "RECENCY_W"),
    "INSTAL_RECENCY_WEIGHTED_LATE_RATE": weighted_group_mean(inst, "SK_ID_CURR", "IS_LATE", "RECENCY_W"),
})

# Last-12-months and last-6-months only.
inst_l12m = inst[inst["INST_AGE_DAYS"] <= 365].groupby("SK_ID_CURR").agg(
    INSTAL_L12M_DAYS_LATE_MEAN=("DAYS_LATE", "mean"),
    INSTAL_L12M_LATE_RATE=("IS_LATE", "mean"),
    INSTAL_L12M_COUNT=("DAYS_LATE", "count"),
)
inst_l6m = inst[inst["INST_AGE_DAYS"] <= 180].groupby("SK_ID_CURR").agg(
    INSTAL_L6M_LATE_RATE=("IS_LATE", "mean"),
)

# Overall (all-time) late-payment rate, needed as the baseline for the trend feature below.
inst_overall_late_rate = inst.groupby("SK_ID_CURR")["IS_LATE"].mean().rename("INSTAL_LATE_RATE_OVERALL")

inst_agg = (
    inst_agg.set_index("SK_ID_CURR")
    .join(inst_recency_feats)
    .join(inst_l12m)
    .join(inst_l6m)
    .join(inst_overall_late_rate)
    .reset_index()
)

# Trend: is the applicant's last 6 months of payment behaviour worse than their
# lifetime average? Positive = getting worse lately - exactly the kind of signal a
# flat aggregate over the applicant's entire history washes out.
inst_agg["INSTAL_LATE_RATE_TREND"] = inst_agg["INSTAL_L6M_LATE_RATE"] - inst_agg["INSTAL_LATE_RATE_OVERALL"]

print("Aggregated installments_payments shape:", inst_agg.shape)
# Save the Aggregated Dataset
inst_agg.to_parquet(f"{OUTPUT_DIR}/agg_installments.parquet", index=False)


# Delete the original large DataFrame.
del inst
# Release unused memory.
gc.collect()

Loaded installments_payments: (13605401, 8)
Aggregated installments_payments shape: (339587, 51)


0

The installments_payments.csv table contains the detailed repayment history for customers' previous loans, recording each installment that was expected and the payment that was actually made. Since a customer can have multiple loans and each loan can have many installments, the same customer may appear many times in this table. The goal is therefore to summarize all these records into one row per customer (SK_ID_CURR).

Before aggregation, the code creates three new features that describe payment behaviour. DAYS_LATE measures how early or late a payment was, with positive values indicating that the payment was late. AMT_PAYMENT_DIFF compares the expected installment amount with the amount actually paid, where a positive value indicates that the customer paid less than required. AMT_PAYMENT_RATIO shows how much of the expected amount was actually paid, with a value of 1 meaning the installment was paid exactly in full.

The data is then grouped by customer, and for each numerical feature the code calculates the mean, maximum, minimum, total, and standard deviation. These statistics capture the customer's typical repayment behaviour, their worst and best payment situations, their overall payment activity, and how consistent their payments have been. The code also counts the number of unique previous loans with installment records and the total number of installment payments.

The code then adds time-aware features because recent payment behaviour can be more useful for predicting current credit risk than very old payment behaviour. It gives greater weight to recent installments when calculating the customer's weighted average days late and weighted late-payment rate. This means that a payment made recently has more influence than one made several years ago.

It also looks specifically at the last 6 and 12 months of payment history. This captures the customer's recent late-payment rate and the number of recent installment records. The code then compares the last 6 months' late-payment rate with the customer's overall historical late-payment rate. This creates a trend feature: a positive value means the customer has been late more often recently than they have been historically, suggesting that their repayment behaviour may be getting worse.

Finally, all the standard and time-aware features are combined into one customer-level table, the columns are given clear INSTAL_ prefixes, and the result is saved as a Parquet file. The large original dataset is then deleted from memory to free up RAM and keep the remaining feature-engineering process efficient.

#### G. Merge everything onto ***application_train*** / ***application_test***

Each secondary table is now one row per `SK_ID_CURR`, so a left-merge won't duplicate any rows. We verify row counts stay constant after every merge as a sanity check.

In [14]:
 # Load the main application file (either train or test)
def build_features(app_path, is_train):
    df = pd.read_csv(app_path)
    # Reduce memory usage by converting columns to smaller data types
    df = downcast(df)
    # Store the original number of rows.
    # This will later be used to verify that merging does not accidentally duplicate or remove customers.
    start_rows = df.shape[0]

# List of Aggregated Feature Tables
   # These datasets were created during the previous preprocessing steps.
    pieces = [
        f'{OUTPUT_DIR}/agg_bureau.parquet',
        f'{OUTPUT_DIR}/agg_pos_cash.parquet',
        f'{OUTPUT_DIR}/agg_credit_card.parquet',
        f'{OUTPUT_DIR}/agg_previous_app.parquet',
        f'{OUTPUT_DIR}/agg_installments.parquet',
    ]

  # Merge Every Aggregated Dataset
 # Loop through each summarized table one at a time and attach it
    for path in pieces:
        piece = downcast(pd.read_parquet(path))       # Optimize memory usage
        # Attach it onto the main table, matching rows by applicant ID
        # "left" means: keep every row in df, even if a person has no matching data in this table (those columns just become blank)
        df = df.merge(piece, on='SK_ID_CURR', how='left')
         # Verify that the number of customers
        # has not changed after merging.
        assert df.shape[0] == start_rows, f"Row count changed after merging {path}!"

       # Delete the temporary dataset
        del piece
        # Release unused memory
        gc.collect()
    # Display Final Dataset Shape
    print(f"{'train' if is_train else 'test'} final shape:", df.shape)
       # Return the Final Dataset
    return df
# Build the Training Dataset
train = build_features(f'{INPUT_DIR}/application_train.csv', is_train=True)
# Build the Testing Dataset
test = build_features(f'{INPUT_DIR}/application_test.csv', is_train=False)

train final shape: (307511, 618)
test final shape: (48744, 616)


This function takes the main application file and combines it with the five summarized tables created earlier, attaching them one at a time using the applicant ID (SK_ID_CURR). This brings together information about each applicant's previous credits, POS/Cash loans, credit cards, previous applications, and installment payments into one dataset.

It starts by loading the main application data and reducing its memory usage using the downcast() function. It then records the original number of rows so that we can check later that the merging process has not accidentally created duplicate applicants or removed any rows.

The function then goes through the five aggregated tables one by one. For each table, it:

- Loads the summarized data and reduces its memory usage.
- Merges it with the main application table using SK_ID_CURR.
- Uses a left merge, meaning every applicant in the main application table is kept, even if they have no records in that particular secondary table. In that case, the newly added features will simply contain missing values.
- Checks that the number of rows has remained exactly the same. Because each aggregated table should contain only one row per applicant, a correct merge should not increase the number of rows. If the number changes, the assert statement stops the process and alerts us that there may be duplicate applicant IDs or another problem with the aggregated data.
- Deletes the temporary table from memory before moving to the next one.

Once all five tables have been merged, the function prints the final dataset size and returns the completed feature table.

Finally, the function is run twice: once for the training dataset and once for the test dataset. This ensures that both datasets go through the same feature-engineering and merging process, giving them the same set of features for machine learning.

#### 8. Align train/test columns and save

One-hot encoding can occasionally produce a category in train that never appears in test (or vice versa). Reindexing test to train's exact column set (filling any gaps with 0) prevents this from silently breaking prediction later.

Therefore, I need to have the training and testing datasets ready for machine learning for the model expects both datasets to have the same features, the code makes sure they match before saving them.

In [15]:

# Remove Unnecessary Columns

# Some datasets may contain columns that are not useful for model training. Remove them if they exist.

for c in ['application_date']:
    if c in train.columns:
        train = train.drop(columns=[c])            # Remove from the training dataset
    if c in test.columns:
        test = test.drop(columns=[c])               # Remove from the testing dataset


# Create the List of Feature Columns

# Get the list of all feature columns — everything except the answer column
# (TARGET) and the ID column (SK_ID_CURR), since those aren't model inputs
feature_cols = [c for c in train.columns if c not in ('TARGET', 'SK_ID_CURR')]
# Reorder the test dataset so its columns match the training dataset exactly.
# Keep SK_ID_CURR first because it is needed later to identify customers.
# If any feature is missing in the test set, create it and fill it with 0.
test = test.reindex(columns=['SK_ID_CURR'] + feature_cols, fill_value=0)


# Verify the Columns Match

# Confirm that the training and testing datasets contain exactly the same feature columns.
# TARGET exists only in the training dataset,so it is excluded from the comparison.
assert list(train.columns.drop('TARGET')) == list(test.columns), "Column mismatch between train/test!"

# Display Dataset Shapes
print("train:", train.shape, "| test:", test.shape)
print("Columns aligned OK.")

# Save the processed datasets as Parquet files.
train.to_parquet(f'{OUTPUT_DIR}/train_final.parquet', index=False)
test.to_parquet(f'{OUTPUT_DIR}/test_final.parquet', index=False)
print("Saved train_final.parquet and test_final.parquet")

train: (307511, 617) | test: (48744, 616)
Columns aligned OK.
Saved train_final.parquet and test_final.parquet


The code first removes any columns that are not needed for machine learning, such as application_date. It then creates a list of the actual input features by leaving out TARGET (the answer we want the model to predict) and SK_ID_CURR (the customer ID, which should not be used to make predictions).

Next, the code makes sure that the test dataset has exactly the same features, in the same order, as the training dataset. If a feature exists in the training data but is missing from the test data, it is added and filled with 0. This is important because the model expects the same inputs when making predictions.

The assert statement then checks that the columns in the training and test datasets match correctly. If there is a mismatch, the code stops immediately instead of allowing an error to occur later during model prediction.

Finally, the cleaned and aligned training and test datasets are saved as Parquet files. This means the processed datasets can be loaded quickly later without having to repeat all the earlier feature engineering and aggregation steps.

In [16]:
print(train.shape)
train.head()

(307511, 617)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,INSTAL_SK_ID_PREV_NUNIQUE,INSTAL_NUM_INSTALMENT_NUMBER_COUNT,INSTAL_RECENCY_WEIGHTED_DAYS_LATE,INSTAL_RECENCY_WEIGHTED_LATE_RATE,INSTAL_L12M_DAYS_LATE_MEAN,INSTAL_L12M_LATE_RATE,INSTAL_L12M_COUNT,INSTAL_L6M_LATE_RATE,INSTAL_LATE_RATE_OVERALL,INSTAL_LATE_RATE_TREND
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,1.0,19.0,-18.732155,0.000000,-17.583334,0.0,12.0,0.0,0.000000,0.000000
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,3.0,25.0,-7.205700,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,1.0,3.0,-7.356328,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,3.0,16.0,-10.061233,0.000000,-6.181818,0.0,11.0,0.0,0.000000,0.000000
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,5.0,66.0,-3.621360,0.006246,-3.615385,0.0,13.0,0.0,0.242424,-0.242424


In [17]:
print(test.shape)
test.head()

(48744, 616)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,INSTAL_SK_ID_PREV_NUNIQUE,INSTAL_NUM_INSTALMENT_NUMBER_COUNT,INSTAL_RECENCY_WEIGHTED_DAYS_LATE,INSTAL_RECENCY_WEIGHTED_LATE_RATE,INSTAL_L12M_DAYS_LATE_MEAN,INSTAL_L12M_LATE_RATE,INSTAL_L12M_COUNT,INSTAL_L6M_LATE_RATE,INSTAL_LATE_RATE_OVERALL,INSTAL_LATE_RATE_TREND
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,2.0,7.0,-15.091597,0.002227,NaN,NaN,NaN,NaN,0.142857,NaN
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,1.0,9.0,-21.375439,0.106319,NaN,NaN,NaN,NaN,0.111111,NaN
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,4.0,155.0,-1.543900,0.135574,-0.619048,0.095238,21.0,0.181818,0.070968,0.110850
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,3.0,113.0,-1.276247,0.071979,-0.631579,0.105263,19.0,0.000000,0.106195,-0.106195
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,1.0,12.0,-11.628451,0.000000,NaN,NaN,NaN,NaN,0.000000,NaN


In [18]:
train.to_csv(f'{OUTPUT_DIR}/train_final.csv', index=False)
test.to_csv(f'{OUTPUT_DIR}/test_final.csv', index=False)
print("Saved CSVs to", OUTPUT_DIR)

Saved CSVs to ./output
